# YouTube Video Summarizer — Ollama + DeepSeek

Summarize YouTube transcripts locally using Llama 3.2 or DeepSeek-R1.

## 1. Install libraries
Run this cell once in your notebook. `%pip` works inside a Jupyter code cell, not PowerShell.

In [ ]:
%pip install -U openai youtube-transcript-api


## 2. Set up Ollama

Run these commands **separately in the VS Code PowerShell terminal**, not in a Python notebook cell. Ollama must be running; if needed, run `ollama serve`. YouTube captions still require an internet connection.


### Download Llama 3.2

```powershell
ollama pull llama3.2
```


### Download DeepSeek-R1 (1.5B)

```powershell
ollama pull deepseek-r1:1.5b
```


### Verify installed models

```powershell
ollama list
```


In [ ]:
from openai import OpenAI

MODELS = {"llama": "llama3.2", "deepseek": "deepseek-r1:1.5b"}
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

choice = input("Choose llama or deepseek [llama]: ").strip().lower() or "llama"
if choice not in MODELS:
    raise ValueError("Choose llama or deepseek")
MODEL = MODELS[choice]
installed = [m.id for m in ollama.models.list().data]
if MODEL not in installed:
    raise RuntimeError(f"Model missing: run ollama pull {MODEL} in PowerShell")
print("Using", MODEL)

## 3. User enters a YouTube URL
When you run this cell, paste the link in the input box and press Enter.

In [ ]:
youtube_url = input("Paste a public YouTube video URL: ").strip()
print("You entered:", youtube_url)

## 4. Extract the video ID and transcript
The transcript is downloaded from YouTube, then sent to your local Ollama model.


In [ ]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

parsed_url = urlparse(youtube_url)
host = (parsed_url.hostname or "").lower()

if host in ("youtu.be", "www.youtu.be"):
    video_id = parsed_url.path.strip("/").split("/")[0]
elif host in ("youtube.com", "www.youtube.com", "m.youtube.com"):
    if parsed_url.path == "/watch":
        video_id = parse_qs(parsed_url.query).get("v", [None])[0]
    elif parsed_url.path.startswith(("/shorts/", "/live/")):
        video_id = parsed_url.path.strip("/").split("/")[1]
    else:
        video_id = None
else:
    video_id = None

if not video_id:
    raise ValueError("Please enter a supported YouTube video URL.")

transcript = YouTubeTranscriptApi().fetch(video_id)
video_text = " ".join(snippet.text for snippet in transcript)
print("Transcript ready! Characters:", len(video_text))
print(video_text[:350])

## 5. Write the prompts
The system prompt defines the summarizer’s behavior; the user prompt gives the specific task.

In [ ]:
system_prompt = """You are a helpful educational YouTube transcript summarizer.
Focus on concepts, important explanations, examples, and actionable takeaways.
Skip greetings, sponsor messages, filler, and repetition.
Do not invent details not present in the transcript.
Organize the result with clear headings and simple language."""

user_prompt = "Summarize the important ideas. Explain each key concept clearly, include examples if discussed, and end with key takeaways."

## 6. Summarize with Llama or DeepSeek

Use the same prompt and transcript with either model. To switch, rerun section 2 and select the other model. DeepSeek-R1 may include reasoning in its response.

In [ ]:
response = ollama.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{user_prompt}\n\nYouTube transcript:\n{video_text}"}
    ]
)
summary = response.choices[0].message.content or "No summary was returned."
print(f"Summary using {MODEL}:\n")
print(summary)

## 7. What I learned

- Run Llama 3.2 and DeepSeek-R1 locally through Ollama
- Switch models using a simple input
- Use an OpenAI-compatible client with a local server
- Extract YouTube captions and summarize them

Try both models on the same video and compare results.